# Preprocessing of earnings calls for DL input

In [1]:
from pathlib import Path
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List, Dict
from tqdm import tqdm
from langdetect import detect
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import re

In [2]:
input_dir = "/Users/jh83999/Dropbox/DL Project - Forecasting/transcripts"
output_dir = "/Users/jh83999/Dropbox/DL Project - Forecasting/source"
os.makedirs(output_dir, exist_ok=True)

TRANSCRIPT_DIR = Path(input_dir)
CHUNK_DIR = Path(output_dir) / "Chunks_model_input"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

#### Preprocessing files

In [ ]:
# ============================================================
# Transcript preprocessing for forecasting task
# Output: one record per earnings call, ready for LLM / DL input
# ============================================================

# === REGEX ===
company_name_patterns = [
    re.compile(pat, re.IGNORECASE) for pat in [
        r"(?:Q\d+\s*&?\s*)?(?:Full Year\s*)?\d{4}\s+(.*?)\s+Earnings\b",
        r"^(.*?)\s+-\s+.*earnings.*-.*conference call",
        r"^(.*?)\s+(?:Fourth|First|Second|Third|Q[1-4]|Qtr|Quarter|Fiscal Year|FY).*?\d{4}.*earnings",
        r"^Q\d+\s+\d{4}\s+(.*?)\s+Q&A",
        r"^(.*?)\s+Fourth Quarter.*Fiscal Year.*\d{4}.*Earnings Conference Call",
        r"^Q\d+\s+\d{4}\s+(.*?)\s+Pre-Recorded Earnings Conference Call"
    ]
]

standard_section_re = re.compile(
    r"""
    -{20,}\s*\n
    (?:Definitions|Disclaimer|Copyright\s+\d{4}\s+Thomson\s+Reuters\.?\s+All\s+Rights\s+Reserved\.?)
    \s*\n
    -{20,}\s*\n
    (.*?)(?=(\n-{80,}\s*\n|$))
""",
    re.DOTALL | re.IGNORECASE | re.VERBOSE
)

speaker_block_re = re.compile(
    r"^-{5,}\s*\n(?P<speaker_line>.+?)\s+\[\d+\]\s*\n-{5,}\s*\n(?P<content>.*?)(?=^-{5,}\s*\n.+?\[\d+\]\s*\n-{5,}|\Z)",
    re.DOTALL | re.MULTILINE
)

suffix_pattern = re.compile(
    r"\s*(\binc(orporated)?\b|\bcorp(oration)?\b|\bltd\b|\bnv\b|\bllc\b|\bplc\b|\bco(mpany)?\b|\bsas\b)\s*",
    re.IGNORECASE
)

conference_section_re = re.compile(
    r"Conference Call Particip\w*\s*=+\s+(.*?)(?=\n={3,}|\n-{3,})",
    re.DOTALL | re.IGNORECASE
)

corporate_section_re = re.compile(
    r"Corporate Particip\w*\s*=+\s+(.*?)(?=\n={3,}|\n-{3,})",
    re.DOTALL | re.IGNORECASE
)

ric_re = re.compile(r"\d{4}-[A-Za-z]{3}-\d{2}-(.+?)-\d{6,}")

datetime_re = re.compile(
    r"(\b(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|SEPT|OCT|NOV|DEC)[A-Z]*\s+\d{1,2},\s+\d{4})\s*/\s*(\d{1,2}:\d{2}(?:AM|PM)\s+GMT)",
    re.IGNORECASE
)

# Optional: remove repetitive operator boilerplate that is usually not informative
boilerplate_line_re = re.compile(
    r"^(operator|coordinator|editor|manager)\b.*$",
    re.IGNORECASE
)

thank_you_only_re = re.compile(
    r"^(thank you|thanks very much|thank you very much)[\.\!\s]*$",
    re.IGNORECASE
)

# === UTILS ===
def normalize_company_name(raw_name: str) -> str:
    raw_name = raw_name.lower().replace(".", "").replace(",", "")
    return suffix_pattern.sub("", raw_name).strip()

def extract_company_name_only(text: str) -> Optional[str]:
    for pat in company_name_patterns:
        m = pat.search(text)
        if m:
            return m.group(1).strip()
    return None

def normalize_for_match(text: str) -> str:
    text = re.sub(r"[.,()]", "", text.lower())
    text = re.sub(r"\bthe\b", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_whitespace(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def clean_block_text(text: str) -> str:
    """
    Light-touch cleaning only:
    - whitespace normalization
    - drop obvious boilerplate single lines
    - keep semantic content intact
    """
    if not text:
        return ""

    lines = [ln.strip() for ln in text.splitlines()]
    cleaned = []

    for ln in lines:
        if not ln:
            cleaned.append("")
            continue
        if boilerplate_line_re.match(ln):
            continue
        cleaned.append(ln)

    text = "\n".join(cleaned)
    text = clean_whitespace(text)
    return text

def extract_corporate_speaker_names(text: str, company_name: str) -> List[str]:
    normalized_company = normalize_for_match(normalize_company_name(company_name))
    section = corporate_section_re.search(text) or conference_section_re.search(text)
    if not section:
        return []

    lines = [line.strip() for line in section.group(1).splitlines() if line.strip()]
    corporate = []
    i = 0

    while i < len(lines):
        line = lines[i]
        if line.startswith("*"):
            name = line[1:].strip()
            affiliation = lines[i + 1].strip() if i + 1 < len(lines) and not lines[i + 1].startswith("*") else ""
            if affiliation:
                i += 2
            else:
                i += 1

            if normalized_company in normalize_for_match(normalize_company_name(affiliation)):
                corporate.append(name)
        else:
            i += 1

    return corporate

def extract_metadata(file_path: Path, full_text: str) -> Dict[str, str]:
    lines = [line.strip() for line in full_text.splitlines() if line.strip()]

    company_line = lines[2] if len(lines) >= 3 else ""
    company_raw = extract_company_name_only(company_line) or "UNKNOWN"
    company = normalize_company_name(company_raw)

    date_line = lines[3] if len(lines) >= 4 else ""
    dt_match = datetime_re.search(date_line) or datetime_re.search(full_text)

    date = dt_match.group(1) if dt_match else "UNKNOWN"
    time = dt_match.group(2) if dt_match else "UNKNOWN"

    ric_match = ric_re.search(file_path.stem)
    ric = ric_match.group(1) if ric_match else "UNKNOWN"

    return {
        "file": file_path.name,
        "doc_id": file_path.stem,
        "ric": ric,
        "company": company,
        "date": date,
        "time_gmt": time
    }

def is_corporate_qa_speaker(speaker_line: str, corporate_speakers: List[str]) -> bool:
    speaker_line_l = speaker_line.lower()

    if "unidentified company representative" in speaker_line_l:
        return True

    for spk in corporate_speakers:
        if spk.lower() in speaker_line_l:
            return True

    return False

def build_model_input(presentation_text: str, qa_text: str) -> str:
    """
    Final pre-formatted text to feed into the next forecasting step.
    Keep explicit section markers so downstream models can exploit structure.
    """
    blocks = []

    if presentation_text:
        blocks.append("[PRESENTATION]\n" + presentation_text)

    if qa_text:
        blocks.append("[Q&A]\n" + qa_text)

    if not blocks:
        return ""

    return "\n\n".join(blocks).strip()

def parse_transcript_sections(full_text: str, company_name: str) -> Dict[str, str]:
    corporate_speakers = extract_corporate_speaker_names(full_text, company_name)

    # Reuters/Refinitiv-style transcripts often separate large sections with =====
    section_split = re.split(r"\n={5,}\n", full_text)

    presentation_blocks = []
    qa_blocks = []

    for section in section_split:
        section_lower = section.lower()

        # Presentation
        if "presentation" in section_lower:
            for match in speaker_block_re.finditer(section):
                content = clean_block_text(match.group("content"))
                if content and not thank_you_only_re.match(content):
                    presentation_blocks.append(content)

        # Q&A: keep ONE full Q&A block, but only retain company-side answers
        # as in your current logic; analyst/firm distinction is removed in output
        elif "questions and answers" in section_lower or "q&a" in section_lower:
            for match in speaker_block_re.finditer(section):
                speaker_line = match.group("speaker_line").strip()
                content = clean_block_text(match.group("content"))

                if not content:
                    continue

                if is_corporate_qa_speaker(speaker_line, corporate_speakers):
                    if not thank_you_only_re.match(content):
                        qa_blocks.append(content)

                # keep unidentified non-trivial answer blocks
                elif "unidentified" in speaker_line.lower() and not content.lower().startswith("thank you"):
                    qa_blocks.append(content)

    presentation_text = clean_whitespace("\n\n".join(presentation_blocks))
    qa_text = clean_whitespace("\n\n".join(qa_blocks))

    # Fallback: if structured parsing failed, keep full cleaned transcript
    if not presentation_text and not qa_text:
        fallback_text = clean_block_text(full_text)
        return {
            "presentation_text": "",
            "qa_text": "",
            "full_text_clean": fallback_text,
            "model_input": fallback_text,
            "has_presentation": 0,
            "has_qa": 0
        }

    model_input = build_model_input(presentation_text, qa_text)
    full_text_clean = clean_whitespace(
        "\n\n".join([x for x in [presentation_text, qa_text] if x])
    )

    return {
        "presentation_text": presentation_text,
        "qa_text": qa_text,
        "full_text_clean": full_text_clean,
        "model_input": model_input,
        "has_presentation": int(bool(presentation_text)),
        "has_qa": int(bool(qa_text))
    }

def preprocess_transcript(file_path: Path) -> Optional[Path]:
    try:
        # skip duplicate download artifacts
        if file_path.name.endswith(" (1).txt"):
            return None

        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        # remove standard header/footer disclaimer sections
        full_text = standard_section_re.sub("", raw_text)
        full_text = clean_whitespace(full_text)

        meta = extract_metadata(file_path, full_text)
        parsed = parse_transcript_sections(full_text, meta["company"])

        # language check on model-relevant text only
        lang_probe = parsed["model_input"][:5000] if parsed["model_input"] else full_text[:5000]
        try:
            lang = detect(lang_probe) if lang_probe.strip() else "unknown"
        except Exception:
            lang = "unknown"

        if lang != "en":
            return None

        record = {
            **meta,
            "language": lang,
            **parsed,
            "n_chars_model_input": len(parsed["model_input"]),
            "n_words_model_input": len(parsed["model_input"].split()) if parsed["model_input"] else 0
        }

        df_chunk = pd.DataFrame([record])
        out_path = CHUNK_DIR / f"{file_path.stem}.parquet"
        df_chunk.to_parquet(out_path, index=False, compression="zstd")

        return out_path

    except Exception as e:
        print(f"Failed: {file_path.name} — {e}")
        return None

# PARALLEL EXECUTION
files = sorted(TRANSCRIPT_DIR.glob("*.txt"))

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(preprocess_transcript, fp): fp for fp in files}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing transcripts"):
        _ = future.result()

Processing transcripts: 100%|██████████| 460580/460580 [2:08:32<00:00, 59.72it/s]  


#### Combine per-transcript file into single document (filter already for S&P 500 membership)

In [3]:
# MERGE  TRANSCRIPTS THAT ARE IN THE S&P 500 AT CALL DATE

# ---------- CONFIG ----------
INDEX_FILE = Path(output_dir) / "SPX_index_leavers_joiners.xlsx"
parquet_files = sorted(CHUNK_DIR.glob("*.parquet"))
output_file = Path(output_dir) / "transcript_model_input_sp500.parquet"

# ============================================================
# HELPERS: SP500 EVENTS -> INTERVALS
def _read_index_events(path, idx_name="SPX"):
    tmp = pd.read_excel(path, header=3)
    tmp.columns = [str(c).strip() for c in tmp.columns]

    need = {"Status", "Code", "Date"}
    missing = need - set(tmp.columns)
    if missing:
        raise KeyError(f"{path.name} is missing columns: {sorted(missing)}")

    tmp = tmp[["Status", "Code", "Date"]].copy()
    tmp["Status"] = tmp["Status"].astype(str).str.strip().str.title()
    tmp["Date"] = pd.to_datetime(tmp["Date"], errors="coerce")
    tmp["ric"] = tmp["Code"].astype(str).str.strip()

    tmp = tmp[
        tmp["Status"].isin(["Joiner", "Leaver"])
        & tmp["Date"].notna()
        & tmp["ric"].ne("")
    ].copy()

    tmp["idx"] = idx_name
    tmp = tmp.rename(columns={"Date": "date", "Status": "status"})
    return tmp[["ric", "date", "status", "idx"]]

def _events_to_intervals(events, sample_start, sample_end):
    intervals = []

    if events.empty:
        return pd.DataFrame(columns=["ric", "start", "end"])

    events = events.sort_values(["ric", "date", "status"]).copy()

    for ric, grp in events.groupby("ric", sort=False):
        active = False
        current_start = None

        for _, row in grp.iterrows():
            status = row["status"]
            dt = pd.Timestamp(row["date"])

            if status == "Joiner":
                if not active:
                    current_start = dt
                    active = True

            elif status == "Leaver":
                if active:
                    intervals.append({
                        "ric": ric,
                        "start": current_start,
                        "end": dt
                    })
                    current_start = None
                    active = False
                else:
                    # Conservative fallback if first observed event is a leaver
                    intervals.append({
                        "ric": ric,
                        "start": sample_start,
                        "end": dt
                    })

        if active and current_start is not None:
            intervals.append({
                "ric": ric,
                "start": current_start,
                "end": sample_end + pd.Timedelta(days=1)
            })

    return pd.DataFrame(intervals)

def _build_interval_lookup(intervals_df):
    lookup = {}
    for ric, grp in intervals_df.groupby("ric", sort=False):
        lookup[ric] = list(zip(grp["start"].tolist(), grp["end"].tolist()))
    return lookup

def _is_member_on_date(ric, dt, lookup):
    if pd.isna(dt):
        return False
    intervals = lookup.get(ric, [])
    for start, end in intervals:
        if start <= dt < end:
            return True
    return False

# ============================================================
# BUILD SP500 LOOKUP
events_spx = _read_index_events(INDEX_FILE, idx_name="SPX")

sample_start = events_spx["date"].min()
sample_end = pd.Timestamp.today().normalize()

spx_intervals = _events_to_intervals(events_spx, sample_start, sample_end)
spx_lookup = _build_interval_lookup(spx_intervals)

# ============================================================
# MERGE ONLY ELIGIBLE TRANSCRIPTS
schema = None
writer = None

files_kept = 0
files_skipped = 0

for file_path in tqdm(parquet_files, desc="Merging SP500 transcripts only"):
    try:
        pf = pq.ParquetFile(file_path)

        # Read only minimal metadata from first row group, only needed columns
        meta_tbl = pf.read_row_group(0, columns=["ric", "date"])

        if meta_tbl.num_rows == 0:
            files_skipped += 1
            continue

        ric = str(meta_tbl.column("ric")[0].as_py()).strip()
        dt = pd.to_datetime(meta_tbl.column("date")[0].as_py(), errors="coerce")

        # Decide once per transcript file
        if not _is_member_on_date(ric, dt, spx_lookup):
            files_skipped += 1
            continue

        # Only now stream the full transcript into output
        for rg in range(pf.num_row_groups):
            table = pf.read_row_group(rg)

            if writer is None:
                schema = table.schema
                writer = pq.ParquetWriter(output_file, schema=schema, compression="zstd")

            writer.write_table(table)

        files_kept += 1

    except Exception as e:
        print(f"Skipping {file_path.name}: {e}")
        files_skipped += 1

if writer is not None:
    writer.close()

/var/folders/sl/6zywmkdd27x0qxk_g2j9jf080000gq/T/ipykernel_89971/1972020282.py:28: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  tmp["Status"] = tmp["Status"].astype(str).str.strip().str.title()
/var/folders/sl/6zywmkdd27x0qxk_g2j9jf080000gq

Built SP500 lookup for 1,277 RICs


Merging SP500 transcripts only: 100%|██████████| 451329/451329 [51:11<00:00, 146.94it/s] 


Saved merged SP500-only model-input file to: /Users/jh83999/Dropbox/DL Project - Forecasting/source/transcript_model_input_sp500.parquet
Files kept:    41,775
Files skipped: 409,554
